In [2]:
import pandas as pd
import numpy as np

# Simulated applicant data — this is the kind of data you'll work with
data = {
    "applicant_id":     [101, 102, 103, 104, 105],
    "age":              [28, 45, 32, 55, 23],
    "monthly_income":   [35000, 82000, 47000, None, 21000],
    "monthly_expenses": [28000, 41000, 39000, 51000, 19500],
    "loan_amount":      [100000, 500000, 200000, 300000, 80000],
    "bill_paid_ontime": [10, 12, 7, 11, 3],   # out of 12 months
    "defaulted":        [0, 0, 1, 0, 1]        # 0 = repaid, 1 = defaulted
}

df = pd.DataFrame(data)
df

,applicant_id,age,monthly_income,monthly_expenses,loan_amount,bill_paid_ontime,defaulted
0,101,28,35000.0,28000,100000,10,0
1,102,45,82000.0,41000,500000,12,0
2,103,32,47000.0,39000,200000,7,1
3,104,55,NaN,51000,300000,11,0
4,105,23,21000.0,19500,80000,3,1


In [3]:
print("Rows, Columns:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nBasic stats:\n")
df.describe()

Rows, Columns: (5, 7)

Column types:
 applicant_id          int64
age                   int64
monthly_income      float64
monthly_expenses      int64
loan_amount           int64
bill_paid_ontime      int64
defaulted             int64
dtype: object

Basic stats:



,applicant_id,age,monthly_income,monthly_expenses,loan_amount,bill_paid_ontime,defaulted
count,5.000000,5.000000,4.000000,5.000000,5.000000,5.000000,5.000000
mean,103.000000,36.600000,46250.000000,35700.000000,236000.000000,8.600000,0.400000
std,1.581139,13.126309,26094.379982,12194.260945,171697.408251,3.646917,0.547723
min,101.000000,23.000000,21000.000000,19500.000000,80000.000000,3.000000,0.000000
25%,102.000000,28.000000,31500.000000,28000.000000,100000.000000,7.000000,0.000000
50%,103.000000,32.000000,41000.000000,39000.000000,200000.000000,10.000000,0.000000
75%,104.000000,45.000000,55750.000000,41000.000000,300000.000000,11.000000,1.000000
max,105.000000,55.000000,82000.000000,51000.000000,500000.000000,12.000000,1.000000


In [4]:
print("Missing values:\n", df.isnull().sum())

# Fill missing income with the median (safer than mean for skewed data)
median_income = df["monthly_income"].median()
df["monthly_income"] = df["monthly_income"].fillna(median_income)

print("\nAfter fix:\n", df.isnull().sum())

Missing values:
 applicant_id        0
age                 0
monthly_income      1
monthly_expenses    0
loan_amount         0
bill_paid_ontime    0
defaulted           0
dtype: int64

After fix:
 applicant_id        0
age                 0
monthly_income      0
monthly_expenses    0
loan_amount         0
bill_paid_ontime    0
defaulted           0
dtype: int64


In [5]:
# Who defaulted?
defaulters = df[df["defaulted"] == 1]
print("Defaulters:\n", defaulters)

# Who earns above 40k and didn't default?
good_earners = df[(df["monthly_income"] > 40000) & (df["defaulted"] == 0)]
print("\nGood earners who repaid:\n", good_earners)

Defaulters:
    applicant_id  age  monthly_income  monthly_expenses  loan_amount  \
2           103   32         47000.0             39000       200000   
4           105   23         21000.0             19500        80000   

   bill_paid_ontime  defaulted  
2                 7          1  
4                 3          1  

Good earners who repaid:
    applicant_id  age  monthly_income  monthly_expenses  loan_amount  \
1           102   45         82000.0             41000       500000   
3           104   55         41000.0             51000       300000   

   bill_paid_ontime  defaulted  
1                12          0  
3                11          0  


In [6]:
# Expense-to-income ratio — a real credit signal
df["expense_ratio"] = df["monthly_expenses"] / df["monthly_income"]

# Bill payment reliability — fraction of months paid on time
df["bill_reliability"] = df["bill_paid_ontime"] / 12

# Loan-to-income ratio
df["loan_to_income"] = df["loan_amount"] / df["monthly_income"]

df[["applicant_id", "expense_ratio", "bill_reliability", "loan_to_income"]]

,applicant_id,expense_ratio,bill_reliability,loan_to_income
0,101,0.800000,0.833333,2.857143
1,102,0.500000,1.000000,6.097561
2,103,0.829787,0.583333,4.255319
3,104,1.243902,0.916667,7.317073
4,105,0.928571,0.250000,3.809524


In [7]:
summary = df.groupby("defaulted").agg(
    avg_income        = ("monthly_income", "mean"),
    avg_expense_ratio = ("expense_ratio", "mean"),
    avg_bill_reliability = ("bill_reliability", "mean"),
    count             = ("applicant_id", "count")
).round(3)

print(summary)

           avg_income  avg_expense_ratio  avg_bill_reliability  count
defaulted                                                            
0           52666.667              0.848                 0.917      3
1           34000.000              0.879                 0.417      2
